# 난수 생성기 직접 구현과 예측

파이썬 표준 라이브러리로 난수 생성기를 구현하고, 수열의 구조와 통계적 성질, 생성 속도를 비교합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/rng_from_scratch.ipynb)

Colab의 기본 CPU 런타임에서 위에서부터 실행하세요. 빠르게 확인하려면 첫 코드 셀에서 `QUICK = True`로 바꾸세요.

## LCG 하위 비트의 주기

같은 LCG 출력에서 하위 비트와 상위 비트를 각각 골라 주기를 비교합니다. 최하위 비트가 한 번씩 번갈아 나오는지도 확인합니다.

In [ ]:
QUICK = False

from collections import Counter
from functools import partial
import math
import random
import secrets
import statistics
import struct
import time


WORD_MASK = (1 << 32) - 1
SEED = 42


class LCG:
    def __init__(self, seed):
        self.state = seed & WORD_MASK

    def next_word(self):
        self.state = (1664525 * self.state + 1013904223) & WORD_MASK
        return self.state


def verdict(passed):
    return "PASS" if passed else "FAIL"


def repeats_at(values, period):
    return values[period:] == values[:-period]


def experiment_lcg():
    rng = LCG(SEED)
    words = [rng.next_word() for _ in range(128)]
    expected = [1083814273, 378494188, 2479403867, 955863294, 1613448261]
    print("시드 42의 첫 5개")
    print("  " + ", ".join(map(str, words[:5])))

    checks = [words[:5] == expected]
    print("비트 수 | 하위 최소 주기 | 상위 같은 주기")
    for width in range(1, 5):
        period = 1 << width
        low = [word & (period - 1) for word in words]
        high = [word >> (32 - width) for word in words]
        observed = next(
            (step for step in range(1, period + 1) if repeats_at(low, step)),
            None,
        )
        high_repeats = repeats_at(high, period)
        print(f"{width:>7} | {str(observed):>14} | {'있음' if high_repeats else '없음'}")
        checks.append(observed == period and not high_repeats)

    lowest = [word & 1 for word in words]
    highest = [word >> 31 for word in words]
    print("하위 1비트 처음 16개 " + "".join(map(str, lowest[:16])))
    print("상위 1비트 처음 16개 " + "".join(map(str, highest[:16])))
    return all(checks) and lowest == [1, 0] * 64


print(f"판정 {verdict(experiment_lcg())}")

## RANDU의 15개 평면

연속한 세 출력을 좌표로 놓고 각 점이 어느 평면에 속하는지 정수 연산으로 확인합니다. 평면별 점 개수를 세어 3차원에서 드러나는 규칙을 봅니다.

In [ ]:
RANDU_MODULUS = 1 << 31


class RANDU:
    def __init__(self, seed):
        self.state = seed % RANDU_MODULUS

    def next_word(self):
        self.state = (65539 * self.state) % RANDU_MODULUS
        return self.state


def experiment_randu(quick):
    count = 100_000 if quick else 200_000
    rng = RANDU(1)
    x, y = rng.next_word(), rng.next_word()
    planes = Counter()
    for _ in range(count):
        z = rng.next_word()
        # (65539 − 3)² = 2³²이므로 a² − 6a + 9는 2³¹의 배수다.
        # 정규화 뒤 9x−6y+z에 해당하는 값을 정수 연산으로 확인한다.
        plane, remainder = divmod(9 * x - 6 * y + z, RANDU_MODULUS)
        if remainder == 0:
            planes[plane] += 1
        x, y = y, z

    matches = sum(planes.values())
    print(f"연속 세 값의 합동식 일치 {matches:,}/{count:,}")
    print(f"정규화한 점이 놓인 평면 {len(planes)}개")
    print("번호 | 점 개수")
    for plane, observed in sorted(planes.items()):
        print(f"{plane:>4} | {observed:>7,}")
    return matches == count and set(planes) == set(range(-5, 10))


print(f"판정 {verdict(experiment_randu(QUICK))}")

## MT19937 직접 구현과 Python random 대조

Python `random`에서 읽은 내부 상태로 직접 구현한 MT19937을 초기화합니다. 두 생성기에서 같은 수의 32비트 정수를 뽑아 하나씩 비교합니다.

In [ ]:
class MT19937:
    N = 624
    M = 397
    MATRIX_A = 0x9908B0DF
    UPPER_MASK = 0x80000000
    LOWER_MASK = 0x7FFFFFFF

    def __init__(self, words, index):
        if len(words) != self.N or not 0 <= index <= self.N:
            raise ValueError("MT19937 상태 크기 또는 인덱스 오류")
        self.words = list(words)
        self.index = index

    @classmethod
    def from_python(cls, generator):
        _, state, _ = generator.getstate()
        return cls(state[:-1], state[-1])

    def twist(self):
        # 뒤쪽에서는 이번 twist에서 갱신한 워드를 다시 참조한다.
        for i in range(self.N):
            following = (i + 1) % self.N
            joined = (self.words[i] & self.UPPER_MASK) | (
                self.words[following] & self.LOWER_MASK
            )
            twisted = joined >> 1
            if joined & 1:
                twisted ^= self.MATRIX_A
            self.words[i] = self.words[(i + self.M) % self.N] ^ twisted
        self.index = 0

    def next_word(self):
        if self.index == self.N:
            self.twist()
        word = self.words[self.index]
        self.index += 1
        word ^= word >> 11
        word ^= (word << 7) & 0x9D2C5680
        word ^= (word << 15) & 0xEFC60000
        return word ^ (word >> 18)

    def random(self):
        # CPython과 똑같이 두 워드에서 상위 27비트와 26비트를 취한다.
        high = self.next_word() >> 5
        low = self.next_word() >> 6
        return ((high << 26) | low) / (1 << 53)


def experiment_mt():
    reference = random.Random(SEED)
    mt = MT19937.from_python(reference)
    count = 1_000 if QUICK else 10_000
    matches = sum(mt.next_word() == reference.getrandbits(32) for _ in range(count))
    print(f"동일한 상태에서 32비트 출력 일치 {matches:,}/{count:,}")
    return matches == count


print(f"판정 {verdict(experiment_mt())}")

## MT19937 상태 저장과 재생

Python `random`의 내부 상태를 직접 저장해 이후 출력을 재생합니다. 상태 워드 624개를 모두 제공했을 때와 623개만 제공했을 때를 비교합니다. [Python random 문서](https://docs.python.org/3/library/random.html)

In [ ]:
def experiment_state_replay():
    reference = random.Random(SEED)
    _, state, _ = reference.getstate()
    words, index = state[:-1], state[-1]
    replay = MT19937(words, index)
    count = 100 if QUICK else 1_000

    word_matches = sum(
        replay.next_word() == reference.getrandbits(32) for _ in range(count)
    )
    float_matches = sum(replay.random() == reference.random() for _ in range(count))
    print(f"상태 워드 {len(words)}개로 재생한 32비트 정수 일치 {word_matches:,}/{count:,}")
    print(f"이어서 재생한 random() 출력 일치 {float_matches:,}/{count:,}")

    try:
        MT19937(words[:-1], index)
    except ValueError as error:
        print(f"상태 워드 {len(words) - 1}개 입력 {error}")
        short_state_rejected = True
    else:
        short_state_rejected = False

    return word_matches == float_matches == count and short_state_rejected


print(f"판정 {verdict(experiment_state_replay())}")

## ChaCha20 RFC 8439 시험 벡터

ChaCha20을 구현해 블록 출력과 암호문을 공개 시험 벡터에 대조합니다. 두 번으로 나누어 키스트림을 읽어도 암호문이 일치하는지 확인합니다. [RFC 8439 §2.3.2 및 §2.4.2](https://www.rfc-editor.org/rfc/rfc8439.html#section-2.3.2)

In [ ]:
def rotate_left(word, count):
    return ((word << count) | (word >> (32 - count))) & WORD_MASK


def quarter_round(words, a, b, c, d):
    words[a] = (words[a] + words[b]) & WORD_MASK
    words[d] = rotate_left(words[d] ^ words[a], 16)
    words[c] = (words[c] + words[d]) & WORD_MASK
    words[b] = rotate_left(words[b] ^ words[c], 12)
    words[a] = (words[a] + words[b]) & WORD_MASK
    words[d] = rotate_left(words[d] ^ words[a], 8)
    words[c] = (words[c] + words[d]) & WORD_MASK
    words[b] = rotate_left(words[b] ^ words[c], 7)


def chacha20_block(key, counter, nonce):
    if len(key) != 32 or len(nonce) != 12:
        raise ValueError("ChaCha20 키는 32바이트, 논스는 12바이트여야 함")
    if not 0 <= counter <= WORD_MASK:
        raise ValueError("ChaCha20 카운터의 32비트 범위 초과")

    initial = (
        struct.unpack("<4I", b"expand 32-byte k")
        + struct.unpack("<8I", key)
        + (counter,)
        + struct.unpack("<3I", nonce)
    )
    words = list(initial)
    for _ in range(10):
        quarter_round(words, 0, 4, 8, 12)
        quarter_round(words, 1, 5, 9, 13)
        quarter_round(words, 2, 6, 10, 14)
        quarter_round(words, 3, 7, 11, 15)
        quarter_round(words, 0, 5, 10, 15)
        quarter_round(words, 1, 6, 11, 12)
        quarter_round(words, 2, 7, 8, 13)
        quarter_round(words, 3, 4, 9, 14)
    return struct.pack(
        "<16I", *((word + start) & WORD_MASK for word, start in zip(words, initial))
    )


class ChaCha20Rng:
    # 교육용. 고정 키·논스 예제와 이 구현을 실사용하지 말 것.
    def __init__(self, key, nonce, counter=0):
        if not 0 <= counter <= WORD_MASK:
            raise ValueError("ChaCha20 초기 카운터의 32비트 범위 초과")
        self.key = key
        self.nonce = nonce
        self.counter = counter
        self.pending = b""

    def read(self, size):
        if size < 0:
            raise ValueError("요청한 바이트 수가 음수")
        if size > len(self.pending) + ((1 << 32) - self.counter) * 64:
            raise OverflowError("ChaCha20 키스트림 소진")

        data = bytearray(self.pending)
        while len(data) < size:
            data.extend(chacha20_block(self.key, self.counter, self.nonce))
            self.counter += 1
        self.pending = bytes(data[size:])
        return bytes(data[:size])

    def next_word(self):
        return int.from_bytes(self.read(4), "little")


def sample_chacha20():
    # 공개 시험 데이터에서 만든 고정 입력.
    return ChaCha20Rng(bytes(range(32)), bytes(range(12)))


def experiment_chacha20():
    key = bytes(range(32))
    block_nonce = bytes.fromhex("000000090000004a00000000")
    expected_block = bytes.fromhex(
        "10f1e7e4d13b5915500fdd1fa32071c4"
        "c7d1f4c733c068030422aa9ac3d46c4e"
        "d2826446079faa0914c2d705d98b02a2"
        "b5129cd1de164eb9cbd083e8a2503c4e"
    )
    block_ok = chacha20_block(key, 1, block_nonce) == expected_block
    print(f"RFC 8439 §2.3.2 블록 64바이트 {verdict(block_ok)}")

    nonce = bytes.fromhex("000000000000004a00000000")
    plaintext = (
        b"Ladies and Gentlemen of the class of '99: If I could offer you "
        b"only one tip for the future, sunscreen would be it."
    )
    expected_ciphertext = bytes.fromhex(
        "6e2e359a2568f98041ba0728dd0d6981"
        "e97e7aec1d4360c20a27afccfd9fae0b"
        "f91b65c5524733ab8f593dabcd62b357"
        "1639d624e65152ab8f530c359f0861d8"
        "07ca0dbf500d6a6156a38e088a22b65e"
        "52bc514d16ccf806818ce91ab7793736"
        "5af90bbf74a35be6b40b8eedf2785e42"
        "874d"
    )
    rng = ChaCha20Rng(key, nonce, counter=1)
    stream = rng.read(13) + rng.read(len(plaintext) - 13)
    ciphertext = bytes(message ^ mask for message, mask in zip(plaintext, stream))
    cipher_ok = len(plaintext) == 114 and ciphertext == expected_ciphertext
    print(f"RFC 8439 §2.4.2 암호문 114바이트 {verdict(cipher_ok)}")
    return block_ok and cipher_ok


print(f"판정 {verdict(experiment_chacha20())}")

## 난수열 통계 검정

전체 비트 빈도(monobit), 블록 빈도(block frequency), 같은 비트가 연속하는 구간(runs), 바이트 카이제곱으로 각 생성기의 출력을 비교합니다. LCG 최하위 비트가 빈도 검정과 runs 검정에서 서로 다른 판정을 받는지 살펴봅니다.

In [ ]:
ALPHA = 0.01


def igamc(shape, x):
    """정규화 상측 불완전 감마 함수 Q(shape, x)."""
    if shape <= 0 or x < 0:
        raise ValueError("감마 함수 인수 범위 오류")
    if x == 0:
        return 1.0

    scale = math.exp(shape * math.log(x) - x - math.lgamma(shape))
    tolerance = 1e-14
    if x < shape + 1:
        # 이 구간에서는 하측 P의 급수를 합산하고 1에서 뺀다.
        term = total = 1.0 / shape
        for i in range(1, 10_001):
            term *= x / (shape + i)
            total += term
            if term <= total * tolerance:
                return min(1.0, max(0.0, 1.0 - scale * total))
    else:
        # 큰 x에서는 수정 렌츠 알고리즘으로 Q의 연분수를 계산한다.
        tiny = 1e-300
        b = x + 1 - shape
        c = 1 / tiny
        d = fraction = 1 / b
        for i in range(1, 10_001):
            numerator = i * (shape - i)
            b += 2
            d = b + numerator * d
            c = b + numerator / c
            if abs(d) < tiny:
                d = math.copysign(tiny, d)
            if abs(c) < tiny:
                c = math.copysign(tiny, c)
            d = 1 / d
            step = c * d
            fraction *= step
            if abs(step - 1) <= tolerance:
                return min(1.0, max(0.0, scale * fraction))
    raise ArithmeticError("불완전 감마 함수가 수렴하지 않음")


def frequency(bits):
    n = len(bits)
    return math.erfc(abs(2 * sum(bits) - n) / math.sqrt(2 * n))


def block_frequency(bits, block_size=128):
    blocks = len(bits) // block_size
    squared = sum(
        (sum(bits[start:start + block_size]) / block_size - 0.5) ** 2
        for start in range(0, blocks * block_size, block_size)
    )
    chi_squared = 4 * block_size * squared
    return igamc(blocks / 2, chi_squared / 2)


def runs(bits):
    n = len(bits)
    proportion = sum(bits) / n
    if abs(proportion - 0.5) >= 2 / math.sqrt(n):
        return 0.0
    observed = 1 + sum(left != right for left, right in zip(bits, memoryview(bits)[1:]))
    expected = 2 * n * proportion * (1 - proportion)
    denominator = 2 * math.sqrt(2 * n) * proportion * (1 - proportion)
    return math.erfc(abs(observed - expected) / denominator)


def byte_chi_squared(data):
    counts = Counter(data)
    expected = len(data) / 256
    chi_squared = sum((counts[value] - expected) ** 2 / expected for value in range(256))
    return igamc(255 / 2, chi_squared / 2)


def bytes_to_bits(data):
    return bytes((value >> shift) & 1 for value in data for shift in range(7, -1, -1))


def bits_to_bytes(bits):
    return bytes(
        sum(bits[start + offset] << (7 - offset) for offset in range(8))
        for start in range(0, len(bits), 8)
    )


def selected_bits(next_word, shift, count):
    return bytes((next_word() >> shift) & 1 for _ in range(count))


def experiment_statistics(quick):
    count = 100_000 if quick else 1_000_000
    byte_count = count // 8
    lcg_low = selected_bits(LCG(SEED).next_word, 0, count)
    lcg_high = selected_bits(LCG(SEED).next_word, 31, count)
    randu_high = selected_bits(RANDU(1).next_word, 30, count)
    python_mt = random.Random(SEED)
    mt_data = b"".join(
        python_mt.getrandbits(32).to_bytes(4, "big")
        for _ in range((byte_count + 3) // 4)
    )[:byte_count]
    sources = (
        ("LCG 하위", lcg_low, bits_to_bytes(lcg_low)),
        ("LCG 상위", lcg_high, bits_to_bytes(lcg_high)),
        ("RANDU 상위", randu_high, bits_to_bytes(randu_high)),
        ("Python MT", bytes_to_bits(mt_data), mt_data),
        ("ChaCha20", None, sample_chacha20().read(byte_count)),
        ("secrets", None, secrets.token_bytes(byte_count)),
    )
    print(f"대상별 {count:,}비트, 유의수준 {ALPHA}")
    print("블록 128비트, 마지막 불완전 블록 제외")
    print("대상별 p값과 판정")
    print("대상             빈도     블록 빈도           런    바이트 χ²")

    results = {}
    for name, bits, data in sources:
        if bits is None:
            bits = bytes_to_bits(data)
        p_values = (frequency(bits), block_frequency(bits), runs(bits), byte_chi_squared(data))
        results[name] = p_values
        cells = " ".join(f"{p:7.3g} {verdict(p >= ALPHA)}" for p in p_values)
        # 한글 한 글자의 터미널 폭은 2칸으로 잡는다.
        width = sum(2 if ord(char) >= 0xAC00 else 1 for char in name)
        print(name.ljust(len(name) + 10 - width) + " " + cells)

    lowest = results["LCG 하위"]
    expected_pattern = lowest[0] >= ALPHA and lowest[1] >= ALPHA and (
        lowest[2] < ALPHA and lowest[3] < ALPHA
    )
    valid = all(0.0 <= p <= 1.0 for row in results.values() for p in row)
    # Q(1/2, x) = erfc(√x)를 써서 급수와 연분수 구간을 대조한다.
    gamma_ok = all(
        math.isclose(igamc(0.5, x), math.erfc(math.sqrt(x)), rel_tol=1e-12)
        for x in (1, 4)
    )
    print(f"LCG 최하위 비트의 예상 패턴 {verdict(expected_pattern)}")
    print(f"불완전 감마 함수의 기준값 대조 {verdict(gamma_ok)}")
    print("secrets 표본과 p값은 실행마다 달라질 수 있음")
    return valid and gamma_ok and expected_pattern


print(f"판정 {verdict(experiment_statistics(QUICK))}")

## 생성 속도

같은 개수의 정수를 생성하는 시간을 잽니다. 워밍업을 제외한 중앙값과 최소·최대 시간을 비교합니다.

In [ ]:
def measure(operation):
    operation()  # 워밍업 1회는 버린다.
    elapsed = []
    for _ in range(5):
        start = time.perf_counter()
        result = operation()
        elapsed.append(time.perf_counter() - start)
    return result, (statistics.median(elapsed), min(elapsed), max(elapsed))


def format_timing(timing):
    median, minimum, maximum = (seconds * 1000 for seconds in timing)
    return f"{median:9.3f}  {minimum:9.3f}~{maximum:.3f}"


def experiment_speed(quick):
    count = 10_000 if quick else 100_000
    factories = (
        ("LCG", lambda: LCG(SEED).next_word),
        ("RANDU", lambda: RANDU(1).next_word),
        ("MT19937", lambda: MT19937.from_python(random.Random(SEED)).next_word),
        ("ChaCha20", lambda: sample_chacha20().next_word),
        ("random", lambda: partial(random.Random(SEED).getrandbits, 32)),
        ("secrets", lambda: partial(secrets.randbits, 32)),
    )

    def generate(factory):
        next_word = factory()
        for _ in range(count):
            value = next_word()
        return value

    print(f"생성기마다 정수 {count:,}개, 단위 ms")
    print("RANDU는 31비트 출력을 그대로 사용")
    print("워밍업 1회 제외, 5회 중앙값과 최소~최대")
    print("생성기           중앙값         최소~최대")
    checks = []
    for name, factory in factories:
        value, timing = measure(lambda: generate(factory))
        print(f"{name:<10} {format_timing(timing)}")
        checks.append(0 <= value <= WORD_MASK and timing[1] >= 0)
    return all(checks)


print(f"판정 {verdict(experiment_speed(QUICK))}")

## Debian OpenSSL 사례와 PID 시드 공간

Debian의 OpenSSL 난수 예측 취약점을 참고해, PID만 시드로 쓰는 모형의 후보 수와 엔트로피 상한을 계산합니다. 출력 비트 수가 커져도 가능한 출력 수는 시드 후보 수를 넘을 수 없습니다. [Debian 보안 공지 DSA-1571-1](https://lists.debian.org/debian-security-announce/2008/msg00152.html)

In [ ]:
pid_space = range(1, 32_769)
seed_count = len(pid_space)
output_bits = 128
output_space = 1 << output_bits

print(f"모형의 PID 시드 후보 {seed_count:,}개")
print(f"시드 엔트로피 상한 {math.log2(seed_count):g}비트")
print(f"{output_bits}비트 출력 공간에서 도달 가능한 비율의 상한 {seed_count / output_space:.3g}")

## 결과 읽는 법

- 통계 검정 통과와 예측 불가능성은 별개입니다. 예를 들어 LCG 최하위 비트는 0과 1의 빈도가 같아도 다음 비트를 알 수 있습니다.
- 유의수준이 `0.01`이면 귀무가설이 참인 무작위 원천에서도, 검정의 근사가 잘 맞을 때 같은 검정을 반복하면 장기적으로 100번에 1번꼴로 `FAIL`이 나올 수 있습니다. 여러 검정을 함께 볼 때는 판정 하나와 전체 결과를 구분해서 읽으세요.
- 속도는 해당 Colab 세션에서 관찰한 값입니다. `secrets` 표본의 p값도 실행마다 달라질 수 있습니다.